<a href="https://colab.research.google.com/github/Efe-Godson/3mtt-stage2-analysis/blob/main/notebooks/deliverable_2_analysis_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3MTT Stage 2: Data Analysis Notebook

This notebook contains:
- data quality audit,
- cleaning & standardisation,
- deduplication,
- at-risk fellow analysis,
- statistical testing,
- and publication-quality visualisations.

## 1. Environment Setup & Library Imports

In [5]:
# CORE LIBRARIES

# Data manipulation
import pandas as pd
import numpy as np

# SQLite database connection
import sqlite3

# Date handling
from datetime import datetime

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 2. Data Loading

Raw datasets are loaded directly from the GitHub repository

In [6]:
# Loading raw data

# Fellows cohort dataset
fellows = pd.read_csv("https://raw.githubusercontent.com/Efe-Godson/3mtt-stage2-analysis/main/raw_data/fellows_cohort.csv")

# Reflection survey dataset
reflection = pd.read_excel("https://raw.githubusercontent.com/Efe-Godson/3mtt-stage2-analysis/main/raw_data/reflection_survey.xlsx", skiprows=1)

# Employer engagement dataset
employer = pd.read_csv("https://raw.githubusercontent.com/Efe-Godson/3mtt-stage2-analysis/main/raw_data/employer_engagement.csv")

# ALC weekly operational logs
alc_logs = pd.read_csv("https://raw.githubusercontent.com/Efe-Godson/3mtt-stage2-analysis/main/raw_data/alc_weekly_log.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


### 2.1 Structural Issue - Reflection Survey File

The reflection survey dataset included an extra merged row above the actual column headers.

The file was loaded using the `skiprows=1` parameter in `pandas.read_excel()` so the correct headers could be captured properly.

This prevented column alignment issues later in the analysis.

## 4. INITIAL DATA FAMILIARIZATION

In [7]:
from IPython.display import display, HTML


# Store all datasets in a dictionary for easier iteration
datasets = {
    "Fellows Cohort": fellows,
    "Reflection Survey": reflection,
    "Employer Engagement": employer,
    "ALC Weekly Logs": alc_logs
}


# Initialize HTML container for grid display
html_content = ""


# Generate overview cards for each dataset
for name, df in datasets.items():

    html_content += f"""

    <div style="
        border:1px solid #d9d9d9;
        border-radius:12px;
        padding:18px;
        margin:10px;
        width:45%;
        display:inline-block;
        vertical-align:top;
        background-color:white;
        box-shadow:2px 2px 10px rgba(0,0,0,0.08);
    ">

        <h2 style="color:#1f4e79;">{name}</h2>

        <hr>

        <p><strong>Rows:</strong> {df.shape[0]}</p>
        <p><strong>Columns:</strong> {df.shape[1]}</p>

        <h4>Column Names</h4>

        <div style="
            background-color:#f7f7f7;
            padding:10px;
            border-radius:6px;
            font-size:13px;
            max-height:120px;
            overflow-y:auto;
        ">
            {", ".join(df.columns)}
        </div>

        <h4 style="margin-top:20px;">Data Types</h4>

        {df.dtypes.to_frame(name='dtype').to_html()}

    </div>

    """


# Render dataset overview grid
display(HTML(html_content))

,dtype
fellow_id,object
first_name,object
last_name,object
state,object
alc_code,object
cohort_number,int64
track,object
enrollment_date,object
completion_status,object
certification_status,object


In [8]:
# Display sample records from each dataset
for name, df in datasets.items():

    print(f"\n{'='*60}")
    print(f"{name.upper()} - SAMPLE RECORDS")
    print(f"{'='*60}")

    display(df.head())


FELLOWS COHORT - SAMPLE RECORDS


,fellow_id,first_name,last_name,state,alc_code,cohort_number,track,enrollment_date,completion_status,certification_status
0,3MTT-F00001,Halima,Musa,niger,ALC-013,3,AI/ML,2024-07-27,complete,certified
1,3MTT-F00002,Grace,Okafor,Ebonyi,ALC-037,4,Cybersecurity,2024-02-20,complete,certified
2,3MTT-F00003,Yusuf,Yakubu,Edo,ALC-041,4,Data Analysis,2024-10-26,complete,certified
3,3MTT-F00004,Ekene,Afolabi,Kaduna,ALC-005,4,AI/ML,2024-08-07,complete,certified
4,3MTT-F00005,Grace,Abubakar,Jigawa,ALC-031,4,Cloud Computing,2024-09-05,complete,certified



REFLECTION SURVEY - SAMPLE RECORDS


,fellow_id,week,response_timestamp,survey_score,phone_number,email
0,3MTT-F00001,1,2024-02-05 16:00:00,2.2,+2348165956164,3mtt.f00001@gmail.com
1,3MTT-F00001,2,2024-02-14 13:00:00,4.9,2347067490250,3mtt.f00001@gmail.com
2,3MTT-F00001,4,2024-03-01 11:00:00,4.4,2349058953690,3mtt.f00001@gmail.com
3,3MTT-F00001,5,2024-03-09 09:00:00,4.9,0816-989-8681,3mtt.f00001@yahoo.com
4,3MTT-F00001,7,2024-03-23 16:00:00,3.6,+2347064394353,3mtt.f00001@yahoo.com



EMPLOYER ENGAGEMENT - SAMPLE RECORDS


,employer_name,sector,state,geopolitical_zone,engagement_type,engagement_date,alc_code,fellows_referred
0,Nestle Nigeria,FMCG,Niger,North-Central,webinar,2024-12-31,ALC-045,0
1,Nestle Nigeria,FMCG,Lagos,South-West,fair,2024-03-10,ALC-016,4
2,Tek Experts,Technology,Borno,North-East,placement,2024-10-08,ALC-030,13
3,Inlaks,Technology,Ekiti,South-West,bootcamp,2024-09-29,ALC-040,0
4,Helium Health,HealthTech,Zamfara,North-West,placement,2024-08-03,ALC-037,13



ALC WEEKLY LOGS - SAMPLE RECORDS


,alc_code,week_number,state,geopolitical_zone,fellows_present,sessions_held,facilitator_name,data_quality_flag
0,ALC-001,1,Adamawa,North-East,15,5,Fatima Aliyu,OK
1,ALC-001,2,Adamawa,North-East,7,2,Fatima Aliyu,LOW_ATTENDANCE
2,ALC-001,3,Adamawa,North-East,10,4,Fatima Aliyu,OK
3,ALC-001,4,Adamawa,North-East,9,3,Fatima Aliyu,OK
4,ALC-001,5,Adamawa,North-East,14,3,Fatima Aliyu,OK


In [10]:
# =========================================================
# 4.4 MISSING VALUE OVERVIEW
# =========================================================


# Analyze missing values across all datasets
for name, df in datasets.items():

    print(f"\n{'='*60}")
    print(f"{name.upper()} - MISSING VALUES")
    print(f"{'='*60}")

    missing_summary = pd.DataFrame({
        "Missing Values": df.isnull().sum(),
        "Missing Percentage": (
            df.isnull().mean() * 100
        ).round(2)
    })

    display(
        missing_summary[
            missing_summary["Missing Values"] > 0
        ].sort_values(
            by="Missing Values",
            ascending=False
        )
    )


FELLOWS COHORT - MISSING VALUES


,Missing Values,Missing Percentage
alc_code,47,4.0



REFLECTION SURVEY - MISSING VALUES


,Missing Values,Missing Percentage



EMPLOYER ENGAGEMENT - MISSING VALUES


,Missing Values,Missing Percentage
alc_code,22,5.12



ALC WEEKLY LOGS - MISSING VALUES


,Missing Values,Missing Percentage
alc_code,20,3.02
